# IVIS - teste do detector no Colab

Rode as células **de cima para baixo**.
1. **Célula 1**: prepara tudo (se pedir, escolha o arquivo `best.pt` que você baixou).
2. **Célula 2**: você sobe fotos de caixas de remédio e vê onde o modelo achou a etiqueta.
3. **Célula 3 (opcional)**: tira UMA foto com a webcam (Chrome, aceite a permissão da câmera).

O Colab não faz vídeo ao vivo; para ao vivo use o `webcam_teste.py` no seu computador.

In [ ]:
!pip install -q ultralytics

import os
import cv2
from google.colab import files
from google.colab.patches import cv2_imshow
from ultralytics import YOLO

PESOS = "/content/runs/detect/train/weights/best.pt"
if not os.path.exists(PESOS):
    print("Escolha o arquivo best.pt (NÃO uma foto), o que você baixou do treino:")
    enviado = files.upload()
    PESOS = list(enviado.keys())[0]
    if not PESOS.lower().endswith(".pt"):
        raise ValueError(f"'{PESOS}' não é um modelo (.pt). Rode esta célula de novo e escolha o best.pt.")

modelo = YOLO(PESOS)
TAM = 512  # o treino usou fotos esticadas para 512x512


def detectar_e_mostrar(caminho, conf=0.4):
    img = cv2.imread(caminho)
    if img is None:
        print("Não consegui abrir", caminho)
        return
    h, w = img.shape[:2]
    # imita o Roboflow: estica a foto para 512x512 antes de detectar
    r = modelo.predict(cv2.resize(img, (TAM, TAM)), imgsz=TAM, conf=conf, verbose=False)[0]

    vis = img.copy()
    n = 0
    for (x1, y1, x2, y2), c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist()):
        x1, x2 = int(x1 * w / TAM), int(x2 * w / TAM)
        y1, y2 = int(y1 * h / TAM), int(y2 * h / TAM)
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 3)
        cv2.putText(vis, f"etiqueta {c:.2f}", (x1, max(25, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        n += 1

    print(f"{caminho}: {n} etiqueta(s) encontrada(s)" if n else f"{caminho}: NENHUMA etiqueta encontrada")
    if w > 900:
        vis = cv2.resize(vis, (900, int(h * 900 / w)))
    cv2_imshow(vis)

print("Pronto! Agora rode a célula 2.")

In [ ]:
# Clique em "Escolher arquivos" e selecione uma ou várias fotos
enviados = files.upload()
for nome in enviados:
    detectar_e_mostrar(nome)

In [ ]:
# Tira UMA foto com a webcam e roda o detector (use o Chrome e permita a câmera)
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode


def tirar_foto(arquivo="foto_webcam.jpg", qualidade=0.9):
    display(Javascript("""
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const botao = document.createElement('button');
      botao.textContent = 'Tirar foto';
      div.appendChild(botao);
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => botao.onclick = resolve);
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    """))
    dados = eval_js(f"takePhoto({qualidade})")
    with open(arquivo, "wb") as f:
        f.write(b64decode(dados.split(",")[1]))
    return arquivo


detectar_e_mostrar(tirar_foto())